# Generate Sample Sales Data

This notebook creates daily sales data for 20 SKUs across 4 categories. It follows the same shape as the M5 Forecasting dataset, so this project runs right away without downloading anything.

When a real dataset is available, replace this notebook's output with the real sales history. Just keep the same column names: date, sku_id, category, units_sold.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(7)

## Set Up Categories and Volatility

Each category has a different demand volatility. Apparel demand jumps around more than Grocery, for example. This matters later because more volatile demand needs more safety stock.

In [2]:
categories = ["Electronics", "Furniture", "Apparel", "Grocery"]
sku_per_category = 5
day_count = 400

category_volatility = {
    "Electronics": 8,
    "Furniture": 4,
    "Apparel": 12,
    "Grocery": 3,
}

start_date = pd.Timestamp("2023-01-01")
dates = start_date + pd.to_timedelta(np.arange(day_count), unit="D")

## Create Daily Sales per SKU

Each SKU gets its own base demand, trend, and weekend pattern, plus random noise sized by its category volatility.

In [3]:
rows = []
sku_counter = 1

for category in categories:
    for _ in range(sku_per_category):
        sku_id = f"SKU-{sku_counter:03d}"
        sku_counter += 1

        base_demand = np.random.uniform(15, 60)
        trend_per_day = np.random.uniform(-0.02, 0.05)
        weekend_boost = np.random.uniform(3, 8)
        volatility = category_volatility[category]

        for day_index in range(day_count):
            day_of_week = dates[day_index].dayofweek
            is_weekend = day_of_week >= 5

            demand = base_demand
            demand += trend_per_day * day_index
            if is_weekend:
                demand += weekend_boost
            demand += np.random.normal(0, volatility)
            demand = max(0, round(demand))

            rows.append({
                "date": dates[day_index],
                "sku_id": sku_id,
                "category": category,
                "units_sold": demand,
            })

sales = pd.DataFrame(rows)
print(sales.shape)
sales.head()

(8000, 4)


,date,sku_id,category,units_sold
0,2023-01-01,SKU-001,Electronics,24
1,2023-01-02,SKU-001,Electronics,44
2,2023-01-03,SKU-001,Electronics,17
3,2023-01-04,SKU-001,Electronics,16
4,2023-01-05,SKU-001,Electronics,35


## Save the Dataset

In [4]:
sales.to_csv("../data/raw/sku_sales.csv", index=False)
print("saved sku_sales.csv with", len(sales), "rows")
print(sales.groupby("category")["units_sold"].agg(["mean", "std"]))

saved sku_sales.csv with 8000 rows
                mean        std
category                       
Apparel      47.8510  15.802457
Electronics  32.1665  10.671257
Furniture    46.8180  14.534115
Grocery      32.5000  10.566259
